<a href="https://colab.research.google.com/github/NielsRogge/Transformers-Tutorials/blob/master/GPT-J-6B/Inference_with_GPT_J_6B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## GPT-J-6B推理

在本笔记本中，我们将使用EleutherAI的[GPT-J-6B模型]进行推理（即生成新文本）(https://github.com/kingoflolz/mesh-transformer-jax/)，这是一个在[The Pile]上训练的60亿参数GPT模型(https://arxiv.org/abs/2101.00027)，一个巨大的公开文本数据集，也由EleutherAI收集。该模型本身是使用JAX和Haiku（后者是JAX之上的神经网络库）在TPUv3s上训练的。

[EleutherAI](https://www.eleuther.ai/)它本身就是一个由人工智能研究人员组成的团队，他们正在进行令人惊叹的人工智能研究（并将所有内容公开并免费使用）。他们还创建了[GPT Neo](https://github.com/EleutherAI/gpt-neo)，它们是较小的GPT变体（分别具有1.25亿、13亿和27亿个参数）。在轮毂上查看他们的型号[此处](https://huggingface.co/EleutherAI).

注意：此笔记本电脑需要至少12.1GB的CPU内存。我个人使用Colab Pro来运行它（并将运行时设置为GPU——高RAM使用率）。不幸的是，Colab的免费版本只提供10GB的RAM，这是不够的。

## 负载模型和分词器
 
首先，我们从中心加载模型。我们选择“float16”版本，这意味着所有参数都使用16位存储，而不是默认的float32位（需要两倍的RAM内存）。我们还将“low_cpu_mem_use”设置为“True”（这是在[本PR]中介绍的）(https://github.com/huggingface/transformers/pull/13466))，以便只将模型加载一次到CPU内存中。

接下来，我们将模型移动到GPU并加载相应的标记器，我们将使用它为模型准备文本。

In [ ]:
import os
import mindnlp
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
if "RANK_TABLE_FILE" in os.environ:
    del os.environ["RANK_TABLE_FILE"]

In [ ]:

from mindnlp.transformers import GPTJForCausalLM, AutoTokenizer




In [ ]:
model = GPTJForCausalLM.from_pretrained("EleutherAI/gpt-j-6B", revision="float16", low_cpu_mem_usage=True, from_pt=True)


tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-j-6B")

## 推理

在这里，我们可以提供一个自定义提示，使用模型的标记器准备该提示（模型所需的唯一输入是`input_ids`）。然后，我们将`input_ids`也移动到GPU，并使用`.generate（）`方法自回归生成令牌。请注意，此方法支持各种解码方法，包括波束搜索和前k采样。所有细节都可以在这篇[博客文章]中找到(https://huggingface.co/blog/how-to-generate).

In [ ]:
prompt = "The Belgian national football team "
input_ids = tokenizer(prompt, return_tensors="ms").input_ids

generated_ids = model.generate(input_ids, do_sample=True, temperature=0.9, max_length=200)
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)

请注意，大型GPT模型最有趣的特性之一是它们能够进行所谓的“少量学习”。这意味着，在文本提示中只给出几个示例，该模型能够快速推广到新的、看不见的示例。因此，您可以使用此模型进行少量镜头文本分类，如下所示：

In [ ]:
prompt = """
Sentence: This movie is very nice.
Sentiment: positive

#####

Sentence: I hated this movie, it sucks.
Sentiment: negative

#####

Sentence: This movie was actually pretty funny.
Sentiment: positive

#####

Sentence: This movie could have been better.
Sentiment: """
input_ids = tokenizer(prompt, return_tensors="ms").input_ids

generated_ids = model.generate(input_ids, do_sample=True, temperature=0.9, max_length=200)
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)

另一个注意事项：由于GPT-J-6B是在The Pile（其中包含大量Github代码）上训练的，因此该模型能够执行代码生成（类似于OpenAI的Codex模型）。这里有一个例子：

In [ ]:
prompt = """Instruction: Generate a Python function that lets you reverse a list.

Answer: """
input_ids = tokenizer(prompt, return_tensors="ms").input_ids

generated_ids = model.generate(input_ids, do_sample=True, temperature=0.9, max_length=200)
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)

## 批量推理

您还可以对批量提示执行推理，如下所示：

In [ ]:
# this is a single input batch with size 3
texts = ["Once there was a man ", "The weather today will be ", "The best football player in the world is "]

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
encoding = tokenizer(texts, padding=True, return_tensors='ms')

generated_ids = model.generate(**encoding, do_sample=True, temperature=0.9, max_length=50)
generated_texts = tokenizer.batch_decode(
    generated_ids, skip_special_tokens=True)

for text in generated_texts:
  print("---------")
  print(text)